<a href="https://colab.research.google.com/github/junseok-jay/AI_lab/blob/main/pipeline/Llama_split_test.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip -q uninstall -y datasets fsspec gcsfs transformers tokenizers huggingface-hub accelerate


In [2]:
!pip -q install --no-cache-dir "numpy<2.0"

import numpy
print("numpy:", numpy.__version__)

numpy: 1.26.4


In [3]:
!pip -q install --no-cache-dir \
  "datasets<3.0" \
  "transformers==4.46.3" \
  "accelerate<1.0" \
  safetensors

import datasets, transformers, torch
print("datasets:", datasets.__version__)
print("transformers:", transformers.__version__)
print("torch:", torch.__version__)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 177.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 112.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.3/527.3 kB 97.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 324.4/324.4 kB 261.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 177.6/177.6 kB 241.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 128.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 93.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.33.0 requires gcsfs!=2025.5.0,>=2023.3.0, which is not installed.
bigframes 2.33.0 requires rich<14,>=12.4.4, but you have rich 14.3.2 which is incompatible.


In [4]:
import datasets, transformers, torch
print("datasets:", datasets.__version__)
print("transformers:", transformers.__version__)
print("torch:", torch.__version__)

The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.


0it [00:00, ?it/s]

datasets: 2.21.0
transformers: 4.46.3
torch: 2.9.0+cu128


In [5]:
from google.colab import userdata

# Colab Secrets에서 HF_TOKEN을 불러옵니다.
# Hugging Face 토큰이 제대로 설정되지 않으면 모델 로드 시 오류가 발생합니다.
HF_TOKEN = userdata.get('hf_token')

In [7]:
from datasets import load_dataset

# Wikipedia (대용량)
ds = load_dataset("wikimedia/wikipedia", "20231101.en", split="train[:1000]")
print(ds[0])

Resolving data files:   0%|          | 0/41 [00:00<?, ?it/s]

{'id': '12', 'url': 'https://en.wikipedia.org/wiki/Anarchism', 'title': 'Anarchism', 'text': 'Anarchism is a political philosophy and movement that is skeptical of all justifications for authority and seeks to abolish the institutions it claims maintain unnecessary coercion and hierarchy, typically including nation-states, and capitalism. Anarchism advocates for the replacement of the state with stateless societies and voluntary free associations. As a historically left-wing movement, this reading of anarchism is placed on the farthest left of the political spectrum, usually described as the libertarian wing of the socialist movement (libertarian socialism).\n\nHumans have lived in societies without formal hierarchies long before the establishment of states, realms, or empires. With the rise of organised hierarchical bodies, scepticism toward authority also rose. Although traces of anarchist ideas are found all throughout history, modern anarchism emerged from the Enlightenment. During

In [8]:
import numpy, torch, importlib
print("numpy:", numpy.__version__)
print("torch:", torch.__version__)
print("cuda:", torch.cuda.is_available(), "gpus:", torch.cuda.device_count())
print("has pipelining:", importlib.util.find_spec("torch.distributed.pipelining") is not None)


numpy: 1.26.4
torch: 2.9.0+cu128
cuda: True gpus: 1
has pipelining: True


In [9]:
from datasets import load_dataset

# 가볍게 일부만
raw = load_dataset("wikimedia/wikipedia", "20231101.en", split="train[:2000]")
print(raw[0].keys())
print(raw[0]["text"][:200])


Resolving data files:   0%|          | 0/41 [00:00<?, ?it/s]

dict_keys(['id', 'url', 'title', 'text'])
Anarchism is a political philosophy and movement that is skeptical of all justifications for authority and seeks to abolish the institutions it claims maintain unnecessary coercion and hierarchy, typi


In [11]:
from transformers import AutoTokenizer
import os

MODEL_ID = "meta-llama/Llama-3.2-3B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, token=HF_TOKEN, use_fast=True)

def wiki_to_chat(ex):
    txt = (ex.get("text") or "").strip()
    if not txt:
        return {"text": ""}

    # 너무 길면 앞부분만
    snippet = txt[:1200]

    messages = [
        {"role": "user", "content": "Summarize the following passage in 1-2 sentences.\n\n" + snippet},
        {"role": "assistant", "content": "Okay."}  # 학습용이면 정답이 필요하지만, 여기선 pipeline example batch용
    ]
    # example batch 만들기만 목적이라 assistant는 더미로 둬도 됨
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
    return {"text": text}

chat_ds = raw.map(wiki_to_chat, remove_columns=raw.column_names).filter(lambda x: len(x["text"]) > 0)
print(chat_ds[0]["text"][:250])


Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Filter:   0%|          | 0/2000 [00:00<?, ? examples/s]

<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 18 Feb 2026

<|eot_id|><|start_header_id|>user<|end_header_id|>

Summarize the following passage in 1-2 sentences.

Anarchism is a politica


In [14]:
from torch.utils.data import DataLoader

# ✅ LLaMA 계열 필수: pad_token 지정
tokenizer.pad_token = tokenizer.eos_token

MAX_LEN = 256

def tok_fn(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        max_length=MAX_LEN,
        padding=False,              # collate에서 패딩
        return_attention_mask=True,
    )

tok_ds = chat_ds.map(tok_fn, batched=True, remove_columns=["text"])

def collate_fn(features):
    # ✅ features(list[dict])를 padding해서 torch 텐서 배치(dict)로 반환
    return tokenizer.pad(
        features,
        padding=True,
        return_tensors="pt"
    )

dl = DataLoader(tok_ds, batch_size=2, shuffle=True, collate_fn=collate_fn)
example_batch = next(iter(dl))

print({k: v.shape for k, v in example_batch.items()})
print(example_batch["input_ids"][:1, :20])
print(example_batch["attention_mask"][:1, :20])


Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

{'input_ids': torch.Size([2, 256]), 'attention_mask': torch.Size([2, 256])}
tensor([[128000, 128000, 128006,   9125, 128007,    271,  38766,   1303,  33025,
           2696,     25,   6790,    220,   2366,     18,    198,  15724,   2696,
             25,    220]])
tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])


In [16]:
import importlib
print("has pipelining:", importlib.util.find_spec("torch.distributed.pipelining") is not None)



has pipelining: True


In [17]:
import os, torch
from transformers import AutoModelForCausalLM

MODEL_ID = "meta-llama/Llama-3.2-3B-Instruct"

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    token=HF_TOKEN,
    device_map=None,
)
model.config.use_cache = False
model.eval()
print("model ready")


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

model ready


In [19]:
from torch.distributed.pipelining import pipeline, SplitPoint

# split 경로 자동 탐색 (보통 model.model.layers.14)
split_path = None
for name, _ in model.named_modules():
    if name.endswith("layers.14"):
        split_path = name
        break
print("split_path:", split_path)

device = torch.device("cuda", 0) if torch.cuda.is_available() else torch.device("cpu")
model = model.to(device)

# mb_args = (example_batch["input_ids"].to(device),)
# mb_kwargs = {"attention_mask": example_batch["attention_mask"].to(device)}

mb_args = ()  # ✅ positional 비움
mb_kwargs = {
    "input_ids": example_batch["input_ids"].to(device),
    "attention_mask": example_batch["attention_mask"].to(device),
}

pipe = pipeline(
    module=model,
    mb_args=mb_args,
    mb_kwargs=mb_kwargs,
    split_spec={split_path: SplitPoint.BEGINNING},  # 2-stage면 split 1개
)

print("✅ pipeline created:", type(pipe))


split_path: model.layers.14


/usr/local/lib/python3.12/dist-packages/torch/distributed/pipelining/_IR.py:1005: FutureWarning: `torch.export.export_for_training` is deprecated and will be removed in PyTorch 2.10. Please use `torch.export.export` instead, which is functionally equivalent.
  ep = torch.export.export_for_training(


RuntimeError: It seems that we cannot capture your model as a full graph. Typical reasons include graph breaks, data/shape-dependent control flow, or missing meta kernels for custom operators. You can use our manual pipeline interfaces, or try to fix the graph breaks, see https://pytorch.org/docs/stable/export.html

### 검증 코드

In [21]:
import os, importlib, torch, numpy

def diag_env():
    print("=== Environment ===")
    print("python:", os.sys.version.split()[0])
    print("torch:", torch.__version__)
    print("cuda:", torch.cuda.is_available(), "gpus:", torch.cuda.device_count())
    print("numpy:", numpy.__version__)
    has_pp = importlib.util.find_spec("torch.distributed.pipelining") is not None
    print("has torch.distributed.pipelining:", has_pp)
    return has_pp

has_pp = diag_env()


=== Environment ===
python: 3.12.12
torch: 2.9.0+cu128
cuda: True gpus: 1
numpy: 1.26.4
has torch.distributed.pipelining: True


In [22]:
import inspect
import torch.nn as nn

def get_submodule_by_name(m, name: str):
    cur = m
    for p in name.split("."):
        cur = getattr(cur, p)
    return cur

def diag_model(model, split_path_hint="model.layers.14"):
    print("\n=== Model structure ===")
    print("model class:", type(model))
    # split_path 확인
    try:
        sm = get_submodule_by_name(model, split_path_hint)
        print("split_path OK:", split_path_hint, "->", type(sm))
    except Exception as e:
        print("split_path FAIL:", split_path_hint, "err:", repr(e))

    # layers.0 forward signature 출력(버전마다 다름)
    # 가능한 후보들에서 layers[0] 찾아봄
    layer0 = None
    candidates = ["model.layers.0", "model.model.layers.0", "layers.0"]
    for c in candidates:
        try:
            layer0 = get_submodule_by_name(model, c)
            print("found layer0:", c, type(layer0))
            break
        except Exception:
            pass
    if layer0 is not None:
        print("layer0.forward signature:", inspect.signature(layer0.forward))
    else:
        print("could not locate a decoder layer at common paths")

    # config 중요한 옵션
    cfg = getattr(model, "config", None)
    if cfg is not None:
        print("\n=== Config ===")
        for k in ["use_cache", "return_dict", "output_hidden_states", "output_attentions"]:
            if hasattr(cfg, k):
                print(f"{k}:", getattr(cfg, k))

diag_model(model, split_path_hint="model.layers.14")



=== Model structure ===
model class: <class 'transformers.models.llama.modeling_llama.LlamaForCausalLM'>
split_path OK: model.layers.14 -> <class 'transformers.models.llama.modeling_llama.LlamaDecoderLayer'>
found layer0: model.layers.0 <class 'transformers.models.llama.modeling_llama.LlamaDecoderLayer'>
layer0.forward signature: (hidden_states: torch.Tensor, attention_mask: Optional[torch.Tensor] = None, position_ids: Optional[torch.LongTensor] = None, past_key_value: Optional[transformers.cache_utils.Cache] = None, output_attentions: Optional[bool] = False, use_cache: Optional[bool] = False, cache_position: Optional[torch.LongTensor] = None, position_embeddings: Optional[Tuple[torch.Tensor, torch.Tensor]] = None, **kwargs) -> Tuple[torch.FloatTensor, Optional[Tuple[torch.FloatTensor, torch.FloatTensor]]]

=== Config ===
use_cache: False
return_dict: True
output_hidden_states: False
output_attentions: False


In [27]:
import torch

def diag_forward_smoke(model, example_batch, device=None):
    print("\n=== Forward smoke test ===")
    if device is None:
        device = torch.device("cuda", 0) if torch.cuda.is_available() else torch.device("cpu")
    model = model.to(device)
    model.eval()

    # LLaMA 계열은 캐시 끄는 게 안전
    if hasattr(model, "config"):
        model.config.use_cache = False
        if hasattr(model.config, "return_dict"):
            model.config.return_dict = False

    with torch.no_grad():
        try:
            out = model(
                input_ids=example_batch["input_ids"].to(device),
                attention_mask=example_batch.get("attention_mask", None).to(device) if "attention_mask" in example_batch else None,
                use_cache=False,
                return_dict=False,
            )
            # out은 tuple일 가능성 높음
            if isinstance(out, tuple):
                print("forward OK. out tuple len:", len(out), "first shape:", getattr(out[0], "shape", None))
            else:
                print("forward OK. out type:", type(out))
            return True
        except Exception as e:
            print("forward FAIL:", type(e).__name__, str(e)[:300])
            return False

ok_forward = diag_forward_smoke(model, example_batch)



=== Forward smoke test ===
forward FAIL: RecursionError maximum recursion depth exceeded


In [24]:
import torch

def diag_export(model, example_batch, device=None):
    print("\n=== torch.export diagnostic ===")
    if device is None:
        device = torch.device("cuda", 0) if torch.cuda.is_available() else torch.device("cpu")
    model = model.to(device).eval()

    if hasattr(model, "config"):
        model.config.use_cache = False
        if hasattr(model.config, "return_dict"):
            model.config.return_dict = False
        if hasattr(model.config, "output_hidden_states"):
            model.config.output_hidden_states = False
        if hasattr(model.config, "output_attentions"):
            model.config.output_attentions = False

    args = ()  # kwargs-only (export 안정성)
    kwargs = {
        "input_ids": example_batch["input_ids"].to(device),
    }
    if "attention_mask" in example_batch:
        kwargs["attention_mask"] = example_batch["attention_mask"].to(device)
    kwargs["use_cache"] = False
    kwargs["return_dict"] = False

    try:
        # PyTorch 2.10에서는 export_for_training 경고가 뜰 수 있음
        ep = torch.export.export(model, args=args, kwargs=kwargs)
        print("export OK. exported_program type:", type(ep))
        return True
    except Exception as e:
        print("export FAIL:", type(e).__name__)
        msg = str(e)
        print(msg[:800])
        return False

ok_export = diag_export(model, example_batch)



=== torch.export diagnostic ===
export FAIL: RecursionError
maximum recursion depth exceeded


# 복구 ㄱㄱ

In [25]:
import os, torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_ID = "meta-llama/Llama-3.2-3B-Instruct"

# tokenizer는 재사용해도 되지만, 깔끔하게 다시 로드
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, token=HF_TOKEN, use_fast=True)
tokenizer.pad_token = tokenizer.eos_token

fresh_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    token=HF_TOKEN,
    device_map=None,
)
fresh_model.config.use_cache = False
fresh_model.config.return_dict = False  # ✅ export/trace friendly
fresh_model.eval()

print("fresh model loaded")


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

fresh model loaded


In [29]:
import inspect
layer0 = model.model.layers[0]  # 너는 model.layers.0 경로가 있으니 model.layers[0]일 수도 있음
print("layer0.forward source:", inspect.getsourcefile(layer0.forward))


layer0.forward source: /usr/local/lib/python3.12/dist-packages/transformers/models/llama/modeling_llama.py


In [31]:
import torch
device = torch.device("cuda", 0) if torch.cuda.is_available() else torch.device("cpu")
fresh_model = fresh_model.to(device)
fresh_model.eval()
fresh_model.config.use_cache = False
fresh_model.config.return_dict = False

with torch.no_grad():
    out = fresh_model(
        input_ids=example_batch["input_ids"].to(device),
        attention_mask=example_batch["attention_mask"].to(device),
        use_cache=False,
        return_dict=False,
    )

print("✅ forward ok, logits shape:", out[0].shape)


✅ forward ok, logits shape: torch.Size([2, 256, 128256])


In [32]:
import torch
try:
    ep = torch.export.export(
        fresh_model,
        args=(),
        kwargs={
            "input_ids": example_batch["input_ids"].to(device),
            "attention_mask": example_batch["attention_mask"].to(device),
            "use_cache": False,
            "return_dict": False,
        },
    )
    print("✅ export ok")
except Exception as e:
    print("❌ export fail:", type(e).__name__, str(e)[:300])


✅ export ok


In [ ]:
pipe_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    token=HF_TOKEN,
    device_map=None,
)
pipe_model.config.use_cache = False
pipe_model.config.return_dict = False
pipe_model.eval()
print("pipe_model reloaded")


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [1]:
import torch
from torch.distributed.pipelining import pipeline, SplitPoint

device = torch.device("cuda", 0) if torch.cuda.is_available() else torch.device("cpu")
pipe_model = pipe_model.to(device).eval()

# export-friendly
pipe_model.config.use_cache = False
pipe_model.config.return_dict = False
pipe_model.config.output_hidden_states = False
pipe_model.config.output_attentions = False

mb_args = ()
mb_kwargs = {
    "input_ids": example_batch["input_ids"].to(device),
    "attention_mask": example_batch["attention_mask"].to(device),
    "use_cache": False,
    "return_dict": False,
}

split_spec = {"model.layers.14": SplitPoint.BEGINNING}

try:
    pipe = pipeline(
        module=pipe_model,
        mb_args=mb_args,
        mb_kwargs=mb_kwargs,
        split_spec=split_spec,
    )
    print("✅ pipeline created:", type(pipe))
except Exception as e:
    print("❌ pipeline failed:", type(e).__name__)
    print(str(e)[:1200])


NameError: name 'pipe_model' is not defined